In [573]:
import os
import subprocess
import xml.etree.ElementTree as ET
from functools import lru_cache

import numpy as np
import open3d as o3d
import rclpy
import rosbag2_py
import yaml
from geometry_msgs.msg import Point
from rclpy.qos import QoSProfile, DurabilityPolicy, ReliabilityPolicy
from rclpy.serialization import deserialize_message
from rosidl_runtime_py.utilities import get_message
from scipy.cluster.vq import kmeans2  # sklearn в окружении собран под другой numpy
from scipy.spatial.transform import Rotation
from sensor_msgs.msg import PointCloud2, PointField
from sensor_msgs_py import point_cloud2
from std_msgs.msg import Header
from visualization_msgs.msg import Marker, MarkerArray

In [574]:
path = '/home/user/wheel_crop'

robot_configs_path = '/home/user/git/robot_configs'
robot = 'charlie'
wheels_ref = 'origin/feature/phantom-points'  # ветка robot_configs с реальными колёсами

# колесо -> два бэга с разным углом поворота и лидар, которым они записаны
wheels = {
    'back_right': dict(bags=['br1', 'br2'], lidar='ld_hs_back'),
    'back_left': dict(bags=['bl1', 'bl2'], lidar='ld_hs_back'),
    'front_right': dict(bags=['fr1', 'fr2'], lidar='ld_hs_front'),
    'front_left': dict(bags=['fl1', 'fl2'], lidar='ld_hs_front'),
}

# кроп для каждого бэга: (min, max); x, y — в base_link, z — высота над полом (base_footprint)
crops = {
    'br1': dict(x=(-1.0, np.inf), y=(-1.0, 1.0), z=(0.04, 0.2)),
    'br2': dict(x=(-1.0, np.inf), y=(-1.0, 1.0), z=(0.04, 0.2)),
    'bl1': dict(x=(-1.0, np.inf), y=(-1.0, 1.0), z=(0.04, 0.2)),
    'bl2': dict(x=(-1.0, np.inf), y=(-1.0, 1.0), z=(0.04, 0.2)),
    'fr1': dict(x=(-np.inf, 1.0), y=(-1.0, 1.0), z=(0.01, 0.2)),
    'fr2': dict(x=(-np.inf, 1.0), y=(-1.0, 1.0), z=(0.01, 0.2)),
    'fl1': dict(x=(-np.inf, 1.0), y=(-1.0, 1.0), z=(0.01, 0.2)),
    'fl2': dict(x=(-np.inf, 1.0), y=(-1.0, 1.0), z=(0.01, 0.2)),
    'axes': dict(x=(-2.0, 2.0), y=(-2.0, 2.0), z=(-1.0, 0.2)),
}

# бэг с облаками левого и правого лидаров, уже в base_link
axes_bag = 'axes'
axes_topics = ['/sensing/ld_hs_left/points/precropped', '/sensing/ld_hs_right/points/precropped']

min_points = 500  # меньше точек после кропа — колеса в бэге нет

frame_id = 'base_link'
axis_scale = 0.4  # длина стрелки, м
axis_colors = [(1.0, 0.0, 0.0), (0.0, 1.0, 0.0), (0.0, 0.4, 1.0)]  # оси PCA 1, 2, 3: R, G, B
intersection_color = (1.0, 1.0, 0.0)  # найденные оси моторов — жёлтые
urdf_colors = {'steer': (1.0, 0.0, 1.0), 'wheel': (0.0, 1.0, 1.0)}  # из URDF: мотор (steer_*) пурпурный, колесо (wheel_*) голубое
estimated_colors = {'motor': intersection_color, 'wheel': (1.0, 0.5, 0.0)}  # найденные: мотор жёлтый, колесо оранжевое

steer_estimates = {}  # колесо -> (origin, axis) найденной оси мотора, заполняют ячейки колёс

In [575]:
@lru_cache
def load_calibration():
    with open(os.path.join(robot_configs_path, robot, 'description/sensor_kit/config/calibration.yaml')) as f:
        return yaml.safe_load(f)


def pose_to_T(p):
    """Поза из calibration.yaml (xyz + rpy, как origin в URDF) -> 4x4."""
    T = np.eye(4)
    T[:3, :3] = Rotation.from_euler('xyz', [p['roll'], p['pitch'], p['yaw']]).as_matrix()
    T[:3, 3] = [p['x'], p['y'], p['z']]
    return T


def lidar_to_base_link_T(lidar_frame):
    """base_link -> sensor_kit_base_link -> лидар."""
    calibration = load_calibration()
    return pose_to_T(calibration['base_link']['sensor_kit_base_link']) @ pose_to_T(calibration['sensor_kit_base_link'][lidar_frame])


def floor_z():
    """z пола (base_footprint) в base_link."""
    return load_calibration()['base_link']['base_footprint']['z']


def git_show(ref, file):
    return subprocess.run(['git', '-C', robot_configs_path, 'show', f'{ref}:{file}'],
                          check=True, capture_output=True, text=True).stdout


@lru_cache
def load_urdf_joints():
    """child link -> joint. base_link -> sim__bottom_1 лежит в модели симулятора, steer_* и wheel_* — в vehicle.xacro робота."""
    joints = {}
    for file in ['simulator/description/urdf/model.xacro', f'{robot}/description/urdf/vehicle.xacro']:
        for j in ET.fromstring(git_show(wheels_ref, file)).iter('joint'):
            joints[j.find('child').get('link')] = j
    return joints


def joint_origin_T(j):
    o = j.find('origin')
    rpy = [float(v) for v in o.get('rpy', '0 0 0').split()]
    xyz = [float(v) for v in o.get('xyz', '0 0 0').split()]
    return pose_to_T(dict(zip(['roll', 'pitch', 'yaw', 'x', 'y', 'z'], rpy + xyz)))


def link_to_base_link_T(link):
    """Поза линка в base_link при нулевых углах всех шарниров."""
    joints = load_urdf_joints()
    T_link = np.eye(4)
    while link != 'base_link':
        j = joints[link]
        T_link = joint_origin_T(j) @ T_link
        link = j.find('parent').get('link')
    return T_link


def urdf_steer_axis(wheel, kind='steer'):
    """Ось поворота колеса из URDF в base_link: (origin, axis).

    axis — axis шарнира steer_*; origin — начало линка steer_* (kind='steer', мотор) или wheel_* (kind='wheel', колесо).
    """
    steer_link = f'steer_{wheel}'
    Ts = link_to_base_link_T(steer_link)
    axis = Ts[:3, :3] @ np.array([float(v) for v in load_urdf_joints()[steer_link].find('axis').get('xyz').split()])
    origin = link_to_base_link_T(f'{kind}_{wheel}')[:3, 3]
    return origin, axis


def wheel_offset_from_motor(wheel):
    """Горизонтальное смещение wheel_* от оси steer_* в base_link при нулевом повороте (колесо жёстко висит на моторе)."""
    steer_origin, _ = urdf_steer_axis(wheel, 'steer')
    wheel_origin, _ = urdf_steer_axis(wheel, 'wheel')
    offset = wheel_origin - steer_origin
    offset[2] = 0.0
    return offset


def wheel_from_motor(wheel, motor_origin, motor_axis):
    """Ось колеса относительно найденной оси мотора: URDF-смещение steer_* -> wheel_*,
    повёрнутое так, чтобы URDF-ось мотора совпала с найденной. Возвращает (origin, axis)."""
    steer_origin, steer_axis = urdf_steer_axis(wheel, 'steer')
    wheel_origin, _ = urdf_steer_axis(wheel, 'wheel')
    R, _ = Rotation.align_vectors([motor_axis], [steer_axis])
    return motor_origin + R.apply(wheel_origin - steer_origin), motor_axis


def steer_joint_origin(wheel, motor_origin, motor_axis):
    """Новый origin шарнира steer_* по найденной оси мотора: (parent, child, xyz, rpy, old_xyz, old_rpy).

    xyz — точка найденной оси на высоте steer_* из URDF (высоту облака не определяют), в системе родителя;
    rpy — минимальный поворот, переводящий axis шарнира в найденную ось.
    """
    j = load_urdf_joints()[f'steer_{wheel}']
    parent, child = j.find('parent').get('link'), j.find('child').get('link')
    urdf_origin, _ = urdf_steer_axis(wheel, 'steer')
    axis = motor_axis / np.linalg.norm(motor_axis)
    origin = motor_origin + (urdf_origin[2] - motor_origin[2]) / axis[2] * axis

    T_parent = link_to_base_link_T(parent)
    R_parent = T_parent[:3, :3]
    xyz = R_parent.T @ (origin - T_parent[:3, 3])
    joint_axis = np.array([float(v) for v in j.find('axis').get('xyz').split()])
    R_corr, _ = Rotation.align_vectors([R_parent.T @ axis], [joint_axis])
    rpy = R_corr.as_euler('xyz')
    o = j.find('origin')
    return parent, child, xyz, rpy, o.get('xyz', '0 0 0'), o.get('rpy', '0 0 0')

In [576]:
def read_messages(input_bag: str):
    reader = rosbag2_py.SequentialReader()
    reader.open(
        rosbag2_py.StorageOptions(uri=input_bag, storage_id="mcap"),
        rosbag2_py.ConverterOptions(
            input_serialization_format="cdr", output_serialization_format="cdr"
        ),
    )

    topic_types = reader.get_all_topics_and_types()

    def typename(topic_name):
        for topic_type in topic_types:
            if topic_type.name == topic_name:
                return topic_type.type
        raise ValueError(f"topic {topic_name} not in bag")

    while reader.has_next():
        topic, data, timestamp = reader.read_next()
        msg_type = get_message(typename(topic))
        msg = deserialize_message(data, msg_type)
        yield topic, msg, timestamp
    del reader


def load_bag(bag_path, T):
    """Все кадры бэга в одном облаке, переведённом на T. Возвращает (pts, intensities)."""
    all_point_clouds = []
    all_intensities = []

    for topic, msg, timestamp in read_messages(bag_path):
        raw_data = np.frombuffer(msg.data, dtype=np.uint8)

        num_points = len(raw_data) // 24
        points_raw = raw_data[:num_points * 24].reshape(-1, 24)

        xyz_bytes = points_raw[:, :12].flatten()

        xyz = xyz_bytes.view(dtype=np.float32).reshape(-1, 3)

        all_point_clouds.append(xyz)
        all_intensities.append(points_raw[:, 12])  # intensity: uint8, offset 12

    all_point_clouds = np.vstack(all_point_clouds, dtype=np.float64)
    all_intensities = np.concatenate(all_intensities)

    pts = all_point_clouds @ T[:3, :3].T + T[:3, 3]
    return pts, all_intensities


def first_clouds(bag_path, topics):
    """Первое облако из каждого топика: topic -> (pts, intensities). Облака уже в base_link."""
    clouds = {}
    for topic, msg, timestamp in read_messages(bag_path):
        if topic not in topics or topic in clouds:
            continue
        assert msg.header.frame_id == frame_id, msg.header.frame_id
        points_raw = np.frombuffer(msg.data, dtype=np.uint8).reshape(-1, msg.point_step)
        xyz = points_raw[:, :12].copy().view(np.float32).astype(np.float64)
        clouds[topic] = (xyz, points_raw[:, 12])  # intensity: uint8, offset 12
        if len(clouds) == len(topics):
            break
    return clouds

In [577]:
def crop(pts, intensities, box, floor_z):
    """box: dict x/y/z -> (min, max), границы не включаются; z считается от пола (floor_z в base_link)."""
    heights = pts - [0.0, 0.0, floor_z]
    cropped_mask = np.ones(len(pts), dtype=bool)
    for i, ax in enumerate('xyz'):
        lo, hi = box[ax]
        cropped_mask &= (heights[:, i] > lo) & (heights[:, i] < hi)

    return pts[cropped_mask], intensities[cropped_mask]


def kmeans_filter(pts, intensities, n_clusters=4, min_cluster_frac=0.01, mad_k=3.0):
    """Выбросы по kmeans.

    min_cluster_frac: кластеры меньше этой доли точек считаем выбросами
    mad_k: порог — медиана + mad_k * sigma расстояния до центра внутри кластера
    """
    centroids, labels = kmeans2(pts, n_clusters, minit='++', seed=0)
    dist = np.linalg.norm(pts - centroids[labels], axis=1)

    inlier_mask = np.zeros(len(pts), dtype=bool)
    for i in range(n_clusters):
        m = labels == i
        if not m.any():
            continue
        med = np.median(dist[m])
        sigma = 1.4826 * np.median(np.abs(dist[m] - med))
        inlier_mask[m] = dist[m] <= med + mad_k * sigma
    inlier_mask &= np.bincount(labels, minlength=n_clusters)[labels] >= min_cluster_frac * len(pts)

    print(f'kmeans: {len(pts)} -> {inlier_mask.sum()} points ({(~inlier_mask).sum()} outliers)')
    return pts[inlier_mask], intensities[inlier_mask]


def compute_pca(pts):
    """Возвращает (center, eigvals, axes); оси — столбцы axes, по убыванию дисперсии."""
    center = pts.mean(axis=0)
    eigvals, eigvecs = np.linalg.eigh(np.cov((pts - center).T))
    order = np.argsort(eigvals)[::-1]  # по убыванию дисперсии
    eigvals, axes = eigvals[order], eigvecs[:, order]

    # знак собственного вектора произволен: направляем крупнейшую компоненту в плюс,
    # третью ось берём как cross, чтобы базис был правым (det = +1)
    for i in range(2):
        if axes[np.argmax(np.abs(axes[:, i])), i] < 0:
            axes[:, i] *= -1
    axes[:, 2] = np.cross(axes[:, 0], axes[:, 1])

    print('center:', center.round(4))
    for i in range(3):
        print(f'axis {i + 1}: {axes[:, i].round(4)}  sigma = {np.sqrt(eigvals[i]):.4f} m')
    return center, eigvals, axes


def compare(bags, a, b):
    """Сравнение систем координат PCA двух бэгов."""
    print(f'=== {bags[0]} vs {bags[1]}')
    print('center diff, m:', (b['center'] - a['center']).round(4))
    for i in range(3):
        cos = np.clip(abs(a['axes'][:, i] @ b['axes'][:, i]), 0, 1)
        print(f'axis {i + 1} angle: {np.degrees(np.arccos(cos)):.2f} deg')


def closest_points(p1, d1, p2, d2):
    """Ближайшие точки двух прямых p + s * d: (q1, s, q2, t)."""
    w = p1 - p2
    a, b, c = d1 @ d1, d1 @ d2, d2 @ d2
    d, e = d1 @ w, d2 @ w
    denom = a * c - b * b  # 0, если прямые параллельны
    s = (b * e - c * d) / denom
    t = (a * e - b * d) / denom
    return p1 + s * d1, s, p2 + t * d2, t


def shift_to_motor(center, direction, wheel_offset):
    """Линия колеса -> параллельная линия через ось мотора.

    wheel_offset — смещение колеса от оси мотора при нулевом повороте (колесо вдоль x);
    поворачиваем его на угол колеса в этом бэге и вычитаем из точки линии.
    """
    yaw = np.arctan2(direction[1], direction[0])
    return center - Rotation.from_euler('z', yaw).apply(wheel_offset)


def estimate_steer_axis(a, b, wheel_offset):
    """Ось мотора по двум бэгам: (origin, axis, lines).

    Линии колеса (оси 1 PCA) смещены от оси мотора на wheel_offset, поэтому сначала сдвигаем их к мотору,
    origin — пересечение сдвинутых линий, axis — нормаль к их плоскости, направлена вверх.
    lines — [(start, end)] сдвинутые линии от сдвинутых центров облаков до пересечения.
    """
    u1, u2 = a['axes'][:, 0], b['axes'][:, 0]
    c1, c2 = shift_to_motor(a['center'], u1, wheel_offset), shift_to_motor(b['center'], u2, wheel_offset)
    q1, s, q2, t = closest_points(c1, u1, c2, u2)
    origin = (q1 + q2) / 2
    axis = np.cross(u1, u2)
    axis /= np.linalg.norm(axis)
    if axis[2] < 0:
        axis *= -1

    wheel_q1, _, wheel_q2, _ = closest_points(a['center'], u1, b['center'], u2)
    print('wheel lines intersection:', ((wheel_q1 + wheel_q2) / 2).round(4))
    print('motor axis (lines shifted by', wheel_offset[:2].round(3), '):', origin.round(4))
    print(f'gap between lines: {np.linalg.norm(q1 - q2) * 1000:.1f} mm')
    print('steer axis:', axis.round(4), f' tilt from base_link z: {np.degrees(np.arccos(axis[2])):.2f} deg')
    return origin, axis, [(c1, q1), (c2, q2)]

In [578]:
# transient_local: RViz получит данные, даже если подпишется позже публикации
qos = QoSProfile(
    depth=1,
    durability=DurabilityPolicy.TRANSIENT_LOCAL,
    reliability=ReliabilityPolicy.RELIABLE,
)

fields = [
    PointField(name='x', offset=0, datatype=PointField.FLOAT32, count=1),
    PointField(name='y', offset=4, datatype=PointField.FLOAT32, count=1),
    PointField(name='z', offset=8, datatype=PointField.FLOAT32, count=1),
    PointField(name='intensity', offset=12, datatype=PointField.FLOAT32, count=1),
]


def get_node():
    """Нода и паблишеры создаются один раз на ядро и переживают перезапуск ячеек."""
    global rviz_node, publishers
    if not rclpy.ok():
        rclpy.init()
    if 'rviz_node' not in globals():
        rviz_node = rclpy.create_node('wheel_crop_viz')
        publishers = {}
    return rviz_node


def get_publisher(msg_type, topic):
    node = get_node()
    if topic not in publishers:
        publishers[topic] = node.create_publisher(msg_type, topic, qos)
    return publishers[topic]


def make_header():
    return Header(frame_id=frame_id, stamp=get_node().get_clock().now().to_msg())


def to_point(p):
    return Point(x=float(p[0]), y=float(p[1]), z=float(p[2]))


def arrow_marker(header, ns, id, start, end, color):
    m = Marker(header=header, ns=ns, id=id, type=Marker.ARROW, action=Marker.ADD)
    m.points = [to_point(start), to_point(end)]
    m.scale.x, m.scale.y, m.scale.z = 0.005, 0.01, 0.015  # толщина стрелки, ширина и длина наконечника
    m.color.r, m.color.g, m.color.b = color
    m.color.a = 1.0
    m.pose.orientation.w = 1.0
    return m


def text_marker(header, ns, id, position, text):
    m = Marker(header=header, ns=ns, id=id, type=Marker.TEXT_VIEW_FACING, action=Marker.ADD, text=text)
    m.pose.position = to_point(position)
    m.pose.orientation.w = 1.0
    m.scale.z = 0.03  # высота текста
    m.color.r = m.color.g = m.color.b = m.color.a = 1.0
    return m


def publish_cloud(topic, pts, intensities):
    cloud_msg = point_cloud2.create_cloud(make_header(), fields, np.c_[pts, intensities].astype(np.float32))
    get_publisher(PointCloud2, topic).publish(cloud_msg)
    print(f'published {pts.shape[0]} points to {topic} in frame "{frame_id}"')


def publish_axes(topic, center, axes, label, brightness=1.0):
    """Три оси PCA стрелками + подпись; brightness < 1 — более тёмные цвета, чтобы различать бэги."""
    header = make_header()
    markers = MarkerArray()
    for i in range(3):
        color = tuple(c * brightness for c in axis_colors[i])
        markers.markers.append(arrow_marker(header, 'pca_axes', i, center, center + axis_scale * axes[:, i], color))
    markers.markers.append(text_marker(header, 'pca_label', 0, center + [0.0, 0.0, 0.05], label))

    get_publisher(MarkerArray, topic).publish(markers)
    print(f'published PCA axes to {topic} in frame "{frame_id}"')


def publish_intersection(topic, lines, origin, axis):
    """lines: [(start, end, brightness)] — оси 1 от центров облаков до пересечения; origin, axis — ось поворота."""
    header = make_header()
    markers = MarkerArray()
    markers.markers.append(Marker(header=header, action=Marker.DELETEALL))  # убрать маркеры прошлых публикаций
    for i, (start, end, brightness) in enumerate(lines):
        m = Marker(header=header, ns='axis1_line', id=i, type=Marker.LINE_STRIP, action=Marker.ADD)
        m.points = [to_point(start), to_point(end)]
        m.scale.x = 0.003  # толщина линии
        m.color.r, m.color.g, m.color.b = (c * brightness for c in axis_colors[0])
        m.color.a = 1.0
        m.pose.orientation.w = 1.0
        markers.markers.append(m)

    end = origin + axis_scale * axis
    markers.markers.append(arrow_marker(header, 'axis1_steer_axis', 0, origin, end, intersection_color))
    markers.markers.append(text_marker(header, 'axis1_label', 0, end + [0.0, 0.0, 0.03],
                                       f'({origin[0]:.3f}, {origin[1]:.3f}, {origin[2]:.3f})'))

    get_publisher(MarkerArray, topic).publish(markers)
    print(f'published steer axis from axis 1 intersection to {topic} in frame "{frame_id}"')


def publish_steer_axes(topic, groups):
    """groups: kind -> (color, {name: (origin, axis)}); все группы одним MarkerArray, namespace <kind>_axis / <kind>_label."""
    header = make_header()
    markers = MarkerArray()
    markers.markers.append(Marker(header=header, action=Marker.DELETEALL))  # убрать маркеры прошлых публикаций
    for kind, (color, axes) in groups.items():
        for i, (name, (origin, axis)) in enumerate(axes.items()):
            end = origin + axis_scale * axis
            markers.markers.append(arrow_marker(header, f'{kind}_axis', i, origin, end, color))
            markers.markers.append(text_marker(header, f'{kind}_label', i, end + [0.0, 0.0, 0.03], name))

    get_publisher(MarkerArray, topic).publish(markers)
    print(f'published steer axes to {topic} in frame "{frame_id}"')

In [579]:
def process_bag(bag, T, brightness):
    """Загрузка, кроп, kmeans, PCA и публикация облака с осями. None, если колеса в кропе нет."""
    print(f'=== {bag}')
    pts, intensities = load_bag(os.path.join(path, bag), T)
    pts, intensities = crop(pts, intensities, crops[bag], floor_z())
    if len(pts) < min_points:
        print(f'only {len(pts)} points after crop, skipping')
        return None
    pts, intensities = kmeans_filter(pts, intensities)
    center, eigvals, axes = compute_pca(pts)

    publish_cloud(f'/wheel_crop/{bag}/points', pts, intensities)
    publish_axes(f'/wheel_crop/{bag}/pca', center, axes, bag, brightness)
    return dict(pts=pts, intensities=intensities, center=center, eigvals=eigvals, axes=axes)


def process_wheel(wheel):
    """Оба бэга колеса -> ось мотора (origin, axis), опубликованная в /wheel_crop/<wheel>/axis1_intersection.

    None, если в одном из бэгов нет точек колеса.
    """
    bags = wheels[wheel]['bags']
    T = lidar_to_base_link_T(wheels[wheel]['lidar'])
    a, b = (process_bag(bag, T, brightness) for bag, brightness in zip(bags, [1.0, 0.5]))
    if a is None or b is None:
        print(f'{wheel}: no wheel points in {bags}, skipping')
        return None

    compare(bags, a, b)
    origin, axis, lines = estimate_steer_axis(a, b, wheel_offset_from_motor(wheel))
    publish_intersection(f'/wheel_crop/{wheel}/axis1_intersection',
                         [(start, end, brightness) for (start, end), brightness in zip(lines, [1.0, 0.5])], origin, axis)
    return origin, axis


def compare_with_urdf(estimates, kinds=('steer', 'wheel')):
    """Сравнение найденных осей мотора и колеса с URDF (steer_* и wheel_*)."""
    for wheel, estimate in estimates.items():
        if estimate is None:
            print(f'{wheel}: no estimate')
            continue
        found = {'steer': estimate, 'wheel': wheel_from_motor(wheel, *estimate)}
        for kind in kinds:
            origin, axis = found[kind]
            urdf_origin, urdf_axis = urdf_steer_axis(wheel, kind)
            angle = np.degrees(np.arccos(np.clip(axis @ urdf_axis, -1, 1)))
            print(f'{kind}_{wheel}: estimated xy {origin[:2].round(3)}  urdf xy {urdf_origin[:2].round(3)}  '
                  f'diff xy {(origin - urdf_origin)[:2].round(3)} m  axis angle {angle:.2f} deg')


def publish_first_clouds(bag, topics):
    """Первое облако каждого топика бэга, с кропом, в /wheel_crop/<bag>/<lidar>/points."""
    for topic, (pts, intensities) in first_clouds(os.path.join(path, bag), topics).items():
        pts, intensities = crop(pts, intensities, crops[bag], floor_z())
        lidar = topic.split('/')[2]  # ld_hs_left / ld_hs_right
        publish_cloud(f'/wheel_crop/{bag}/{lidar}/points', pts, intensities)

In [580]:
steer_estimates['back_right'] = process_wheel('back_right')

=== br1
kmeans: 30529 -> 30122 points (407 outliers)
center: [-0.779  -0.1984 -0.1524]
axis 1: [ 0.9682 -0.2498  0.0131]  sigma = 0.1148 m
axis 2: [0.2483 0.9662 0.069 ]  sigma = 0.0282 m
axis 3: [-0.0299 -0.0635  0.9975]  sigma = 0.0070 m
published 30122 points to /wheel_crop/br1/points in frame "base_link"
published PCA axes to /wheel_crop/br1/pca in frame "base_link"
=== br2
kmeans: 15430 -> 15424 points (6 outliers)
center: [-0.7357 -0.4213 -0.1617]
axis 1: [0.9653 0.2593 0.0321]  sigma = 0.1161 m
axis 2: [-0.2592  0.9658 -0.007 ]  sigma = 0.0860 m
axis 3: [-0.0328 -0.0016  0.9995]  sigma = 0.0058 m
published 15424 points to /wheel_crop/br2/points in frame "base_link"
published PCA axes to /wheel_crop/br2/pca in frame "base_link"
=== br1 vs br2
center diff, m: [ 0.0433 -0.2229 -0.0093]
axis 1 angle: 29.51 deg
axis 2 angle: 29.73 deg
axis 3 angle: 3.55 deg
wheel lines intersection: [-0.3335 -0.3133 -0.1473]
motor axis (lines shifted by [-0.   -0.04] ): [-0.3338 -0.272  -0.1472]
gap 

In [581]:
steer_estimates['back_left'] = process_wheel('back_left')

=== bl1
kmeans: 19363 -> 19360 points (3 outliers)
center: [-0.7202  0.4253 -0.1613]
axis 1: [ 0.9693 -0.2422  0.0415]  sigma = 0.1120 m
axis 2: [0.2422 0.9702 0.0045]  sigma = 0.0790 m
axis 3: [-0.0413  0.0056  0.9991]  sigma = 0.0053 m
published 19360 points to /wheel_crop/bl1/points in frame "base_link"
published PCA axes to /wheel_crop/bl1/pca in frame "base_link"
=== bl2
kmeans: 39244 -> 38842 points (402 outliers)
center: [-0.7812  0.1941 -0.1527]
axis 1: [0.972  0.2347 0.0091]  sigma = 0.1135 m
axis 2: [-0.2346  0.972  -0.0111]  sigma = 0.0294 m
axis 3: [-0.0114  0.0086  0.9999]  sigma = 0.0076 m
published 38842 points to /wheel_crop/bl2/points in frame "base_link"
published PCA axes to /wheel_crop/bl2/pca in frame "base_link"
=== bl1 vs bl2
center diff, m: [-0.061  -0.2312  0.0087]
axis 1 angle: 27.65 deg
axis 2 angle: 27.60 deg
axis 3 angle: 1.72 deg
wheel lines intersection: [-0.2803  0.3152 -0.1453]
motor axis (lines shifted by [0.   0.04] ): [-0.2805  0.2741 -0.1451]
gap be

In [582]:
steer_estimates['front_right'] = process_wheel('front_right')

=== fr1
kmeans: 5269 -> 5267 points (2 outliers)
center: [ 0.754  -0.45   -0.1889]
axis 1: [ 0.9132 -0.4056 -0.0391]  sigma = 0.1074 m
axis 2: [ 0.4054  0.914  -0.012 ]  sigma = 0.0312 m
axis 3: [ 0.0406 -0.0049  0.9992]  sigma = 0.0067 m
published 5267 points to /wheel_crop/fr1/points in frame "base_link"
published PCA axes to /wheel_crop/fr1/pca in frame "base_link"
=== fr2
kmeans: 8103 -> 8102 points (1 outliers)
center: [ 0.7869 -0.1754 -0.189 ]
axis 1: [ 0.977   0.2106 -0.0344]  sigma = 0.1141 m
axis 2: [-0.2113  0.9773 -0.0174]  sigma = 0.0310 m
axis 3: [0.03   0.0242 0.9993]  sigma = 0.0058 m
published 8102 points to /wheel_crop/fr2/points in frame "base_link"
published PCA axes to /wheel_crop/fr2/pca in frame "base_link"
=== fr1 vs fr2
center diff, m: [ 3.290e-02  2.746e-01 -2.000e-04]
axis 1 angle: 36.09 deg
axis 2 angle: 36.12 deg
axis 3 angle: 1.78 deg
wheel lines intersection: [ 0.3485 -0.2699 -0.1725]
motor axis (lines shifted by [-0.   -0.04] ): [ 0.3528 -0.2281 -0.1725]


In [583]:
steer_estimates['front_left'] = process_wheel('front_left')

=== fl1
kmeans: 7884 -> 7884 points (0 outliers)
center: [ 0.7853  0.2197 -0.1879]
axis 1: [ 0.9643 -0.2626 -0.0333]  sigma = 0.1125 m
axis 2: [0.2628 0.9648 0.0041]  sigma = 0.0308 m
axis 3: [ 0.031  -0.0127  0.9994]  sigma = 0.0062 m
published 7884 points to /wheel_crop/fl1/points in frame "base_link"
published PCA axes to /wheel_crop/fl1/pca in frame "base_link"
=== fl2
kmeans: 5708 -> 5708 points (0 outliers)
center: [ 0.7613  0.4964 -0.1872]
axis 1: [ 0.9288  0.3681 -0.0422]  sigma = 0.1063 m
axis 2: [-0.3677  0.9298  0.0153]  sigma = 0.0312 m
axis 3: [0.0449 0.0013 0.999 ]  sigma = 0.0070 m
published 5708 points to /wheel_crop/fl2/points in frame "base_link"
published PCA axes to /wheel_crop/fl2/pca in frame "base_link"
=== fl1 vs fl2
center diff, m: [-0.024   0.2768  0.0007]
axis 1 angle: 36.82 deg
axis 2 angle: 36.82 deg
axis 3 angle: 1.13 deg
wheel lines intersection: [ 0.3572  0.3363 -0.171 ]
motor axis (lines shifted by [0.   0.04] ): [ 0.3596  0.2942 -0.1709]
gap between li

In [584]:
# оси поворота из URDF: через начало линка мотора (steer_*) и колеса (wheel_*)
urdf_axes = {}
for kind, color in urdf_colors.items():
    urdf_axes[kind] = (color, {f'{kind}_{wheel}': urdf_steer_axis(wheel, kind) for wheel in wheels})
    for name, (origin, axis) in urdf_axes[kind][1].items():
        print(f'{name}: origin {origin.round(4)}  axis {axis.round(4)}')
publish_steer_axes('/wheel_crop/steer_axes', urdf_axes)

# найденные оси моторов и оси колёс относительно них
found = {wheel: est for wheel, est in steer_estimates.items() if est is not None}
estimated_axes = {
    'motor': (estimated_colors['motor'], {f'motor_{wheel}': est for wheel, est in found.items()}),
    'wheel': (estimated_colors['wheel'], {f'wheel_{wheel}': wheel_from_motor(wheel, *est) for wheel, est in found.items()}),
}
print()
for kind, (color, axes) in estimated_axes.items():
    for name, (origin, axis) in axes.items():
        print(f'estimated {name}: origin {origin.round(4)}  axis {axis.round(4)}')
publish_steer_axes('/wheel_crop/estimated_axes', estimated_axes)

print('\n=== estimated vs URDF')
compare_with_urdf(steer_estimates)

print(f'\n=== {axes_bag}')
publish_first_clouds(axes_bag, axes_topics)

steer_back_right: origin [-0.29   -0.27   -0.0558]  axis [-0.  0.  1.]
steer_back_left: origin [-0.29    0.27   -0.0558]  axis [-0.  0.  1.]
steer_front_right: origin [ 0.29   -0.27   -0.0557]  axis [-0.  0.  1.]
steer_front_left: origin [ 0.29    0.27   -0.0557]  axis [-0.  0.  1.]
wheel_back_right: origin [-0.29   -0.31    0.0042]  axis [-0.  0.  1.]
wheel_back_left: origin [-0.29    0.31    0.0042]  axis [-0.  0.  1.]
wheel_front_right: origin [ 0.29   -0.31    0.0043]  axis [-0.  0.  1.]
wheel_front_left: origin [0.29   0.31   0.0043]  axis [-0.  0.  1.]
published steer axes to /wheel_crop/steer_axes in frame "base_link"

estimated motor_back_right: origin [-0.3338 -0.272  -0.1472]  axis [-0.0232 -0.0374  0.999 ]
estimated motor_back_left: origin [-0.2805  0.2741 -0.1451]  axis [-0.0257  0.0679  0.9974]
estimated motor_front_right: origin [ 0.3528 -0.2281 -0.1725]  axis [ 0.0377 -0.0116  0.9992]
estimated motor_front_left: origin [ 0.3596  0.2942 -0.1709]  axis [0.0389 0.0163 0.999

In [585]:
# новые origin шарниров steer_* для vehicle.xacro по найденным осям моторов
for wheel, est in steer_estimates.items():
    if est is None:
        print(f'steer_{wheel}: no estimate')
        continue
    parent, child, xyz, rpy, old_xyz, old_rpy = steer_joint_origin(wheel, *est)
    print(f'{parent} -> {child}')
    print(f'  old: <origin xyz="{old_xyz}" rpy="{old_rpy}"/>')
    print(f'  new: <origin xyz="{xyz[0]:.4f} {xyz[1]:.4f} {xyz[2]:.4f}" rpy="{rpy[0]:.4f} {rpy[1]:.4f} {rpy[2]:.4f}"/>')

sim__bottom_1 -> steer_back_right
  old: <origin xyz="-0.27 0.01 -0.29" rpy="0 0 0"/>
  new: <origin xyz="-0.2754 0.0100 -0.3359" rpy="-0.0232 0.0004 0.0374"/>
sim__bottom_1 -> steer_back_left
  old: <origin xyz="0.27 0.01 -0.29" rpy="0 0 0"/>
  new: <origin xyz="0.2801 0.0100 -0.2828" rpy="-0.0257 -0.0009 -0.0679"/>
sim__bottom_1 -> steer_front_right
  old: <origin xyz="-0.27 0.01 0.29" rpy="0 0 0"/>
  new: <origin xyz="-0.2294 0.0100 0.3572" rpy="0.0377 -0.0002 0.0116"/>
sim__bottom_1 -> steer_front_left
  old: <origin xyz="0.27 0.01 0.29" rpy="0 0 0"/>
  new: <origin xyz="0.2961 0.0100 0.3640" rpy="0.0389 0.0003 -0.0163"/>
